In [ ]:
# ── Evaluation — CIFAR-100 / ResNet18 ────────────────────────────────────
# Prereq: run 02_emaskd, 03_proposed, 04_proposed, 05_proposed first
import datetime

SLUG = datetime.datetime.now().strftime('%Y-%m-%d-%H%M') + '-135ep'

REPO   = 'https://github.com/almaas-izdihar/4167a324fe'
BRANCH = 'experiment/b2-sigmoid-additive'
DIR    = '/content/ema-skd'

print(f'Slug: {SLUG}')

In [ ]:
# ── Clone repo (pulls latest logs pushed by 02, 03, 04, 05) ──────────────
import subprocess, os, glob
if not os.path.exists(DIR):
    subprocess.run(f'git clone {REPO} {DIR}', shell=True, check=True)
subprocess.run(f'git -C {DIR} fetch origin {BRANCH}', shell=True, check=True)
subprocess.run(f'git -C {DIR} checkout {BRANCH}',     shell=True, check=True)
subprocess.run(f'git -C {DIR} pull origin {BRANCH}',  shell=True, check=True)
os.chdir(DIR)

def latest(pattern):
    files = sorted(glob.glob(pattern))
    assert files, f'No file found: {pattern}'
    return files[-1]

EMASKD_LOG = latest(f'{DIR}/results/02_emaskd/*_log.txt')
LOG_03     = latest(f'{DIR}/results/03_proposed/*_log.txt')  # hard_gate
LOG_04     = latest(f'{DIR}/results/04_proposed/*_log.txt')  # v3 fixed (true_class_weight)
LOG_05     = latest(f'{DIR}/results/05_proposed/*_log.txt')  # B2 sigmoid additive
OUT_DIR    = f'{DIR}/results'

print('02_emaskd (baseline) :', EMASKD_LOG)
print('03_proposed (hard_gate):', LOG_03)
print('04_proposed (v3 fixed) :', LOG_04)
print('05_proposed (B2 sigmoid):', LOG_05)

In [ ]:
# ── Parse logs ───────────────────────────────────────────────────────────
import re
import pandas as pd

def parse_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[val]' not in line: continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1).rstrip('%')) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep: continue
            rows.append({'epoch': int(ep.group(1)), 'top1': g('val_top1_acc'),
                         'top5': g('val_top5_acc'), 'val_loss': g('val_loss'),
                         'ece': g('ECE'), 'aurc': g('AURC'), 'eaurc': g('EAURC')})
    return pd.DataFrame(rows).set_index('epoch')

def parse_train_log(path):
    rows = []
    with open(path) as f:
        for line in f:
            if '[train]' not in line: continue
            def g(key):
                m = re.search(rf'\[{key} ([^\]]+)\]', line)
                return float(m.group(1).rstrip('%')) if m else None
            ep = re.search(r'\[Epoch (\d+)\]', line)
            if not ep: continue
            rows.append({'epoch': int(ep.group(1)), 'train_loss': g('train_loss'),
                         'train_top1': g('train_top1_acc'), 'gate_pct': g('gate'),
                         'tau': g('tau'), 'beta_eff': g('beta_eff'), 'w_run': g('w_run')})
    return pd.DataFrame(rows).set_index('epoch')

df_ema       = parse_log(EMASKD_LOG)
df_03        = parse_log(LOG_03)
df_04        = parse_log(LOG_04)
df_05        = parse_log(LOG_05)
df_ema_train = parse_train_log(EMASKD_LOG)
df_03_train  = parse_train_log(LOG_03)
df_04_train  = parse_train_log(LOG_04)
df_05_train  = parse_train_log(LOG_05)

print(f'02_emaskd : {len(df_ema)} epochs')
print(f'03 hard_gate : {len(df_03)} epochs')
print(f'04 v3 fixed  : {len(df_04)} epochs')
print(f'05 B2 sigmoid: {len(df_05)} epochs')

In [ ]:
# ── Summary table (final epoch) ──────────────────────────────────────────
last_ema = df_ema.iloc[-1]
last_03  = df_03.iloc[-1]
last_04  = df_04.iloc[-1]
last_05  = df_05.iloc[-1]

summary = pd.DataFrame({
    'Metric':    ['Top-1 (%)', 'Top-5 (%)', 'Val Loss', 'ECE (↓)', 'AURC (↓)', 'EAURC (↓)'],
    'EMA-SKD':   [last_ema.top1, last_ema.top5, last_ema.val_loss, last_ema.ece, last_ema.aurc, last_ema.eaurc],
    'HardGate':  [last_03.top1,  last_03.top5,  last_03.val_loss,  last_03.ece,  last_03.aurc,  last_03.eaurc],
    'v3Fixed':   [last_04.top1,  last_04.top5,  last_04.val_loss,  last_04.ece,  last_04.aurc,  last_04.eaurc],
    'B2Sigmoid': [last_05.top1,  last_05.top5,  last_05.val_loss,  last_05.ece,  last_05.aurc,  last_05.eaurc],
})
print(summary.to_string(index=False, float_format=lambda x: f'{x:.3f}'))

print('\nΔ from EMA-SKD baseline:')
for col in ['HardGate', 'v3Fixed', 'B2Sigmoid']:
    deltas = (summary[col] - summary['EMA-SKD']).values
    line = f'  {col:10s}'
    for i, m in enumerate(summary['Metric']):
        line += f'  {m}: {deltas[i]:+.3f}'
    print(line)

In [ ]:
# ── Training curves — Top-1, Val Loss, ECE, AURC ─────────────────────────
import matplotlib.pyplot as plt, matplotlib.ticker as ticker

panels = [
    ('top1',     'Top-1 Accuracy (%)'),
    ('val_loss', 'Val Loss'),
    ('ece',      'ECE (↓)'),
    ('aurc',     'AURC (↓)'),
]

variants = [
    ('EMA-SKD',     df_ema, 'steelblue',    'o'),
    ('HardGate',    df_03,  'darkorange',   's'),
    ('v3 Fixed',    df_04,  'mediumseagreen', '^'),
    ('B2 Sigmoid',  df_05,  'crimson',      'D'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Variants Comparison — CIFAR-100 / ResNet18', fontsize=13)
for ax, (col, title) in zip(axes.flat, panels):
    for label, df, color, marker in variants:
        ax.plot(df.index, df[col], label=label, color=color, marker=marker,
                markevery=max(1, len(df)//10), linewidth=1.5)
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/eval_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR}/eval_curves.png')

# Final-epoch delta vs baseline
print('\nFinal epoch vs EMA-SKD baseline:')
for col, title in panels:
    b = df_ema[col].iloc[-1]
    print(f'  {title:22s}  EMA={b:.3f}'
          f'  03_HG={df_03[col].iloc[-1]:.3f} ({df_03[col].iloc[-1]-b:+.3f})'
          f'  04_v3={df_04[col].iloc[-1]:.3f} ({df_04[col].iloc[-1]-b:+.3f})'
          f'  05_B2={df_05[col].iloc[-1]:.3f} ({df_05[col].iloc[-1]-b:+.3f})')

In [ ]:
# ── Training dynamics ─────────────────────────────────────────────────────
import matplotlib.pyplot as plt

variants_train = [
    ('EMA-SKD',    df_ema_train, 'steelblue'),
    ('HardGate',   df_03_train,  'darkorange'),
    ('v3 Fixed',   df_04_train,  'mediumseagreen'),
    ('B2 Sigmoid', df_05_train,  'crimson'),
]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Training Dynamics — CIFAR-100 / ResNet18', fontsize=13)
for ax, col, title in [(ax1, 'train_loss', 'Train Loss'),
                        (ax2, 'train_top1', 'Train Top-1 Acc (%)')]:
    for label, df, color in variants_train:
        ax.plot(df.index, df[col], label=label, color=color, linewidth=1.5)
    ax.set_title(title); ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUT_DIR}/train_dynamics.png', dpi=120, bbox_inches='tight')
plt.show()
print(f'Saved: {OUT_DIR}/train_dynamics.png')